# 🏎️ Identifying Value Bets in Formula 1
**DSA 210 – Introduction to Data Science | Spring 2026**
**Author:** Hakan Osman Taşkın

---

## Project Overview

This project builds a machine learning system to predict F1 podium finishers and identify **Expected Value (EV)** betting opportunities. By comparing model-derived probabilities (built from qualifying telemetry) against bookmaker market odds, this study aims to detect cases where the market systematically undervalues a driver's chances.

**Core hypothesis:** Pre-race telemetry data — qualifying grid position, lap time delta to pole, and straight-line top speed — contains measurable signals that the betting market fails to fully price in.

### Full Analytical Narrative

| Step | Description | Why |
|---|---|---|
| 1 | Load & describe data | Understand structure, spot quality issues |
| 2 | Clean data | Fix errors before any analysis |
| 3 | Calibrate bookmaker odds | Remove overround bias from market probabilities |
| 4 | EDA | Find patterns that motivate hypotheses |
| 5 | Normality checks | Choose correct statistical tests |
| 6 | Hypothesis testing (H1, H2, H3) | Formally validate observed patterns |
| 7 | ML modelling | Quantify telemetry predictive power |
| 8 | EV simulation | Identify market inefficiencies |
| 9 | 2025 live predictions | Apply model to current season |
| 10 | Limitations & conclusions | Honest assessment of findings |

---

## 1. Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from scipy.optimize import brentq
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.dummy import DummyClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.metrics import (
    roc_auc_score, average_precision_score,
    ConfusionMatrixDisplay, roc_curve, precision_recall_curve
)
from numbers_parser import Document
import warnings

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120

def fix_odds(val):
    """Fix Turkish locale: Numbers stores 1.935 as 1935.0. Divide floats by 1000."""
    if isinstance(val, float):   return round(val / 1000, 3)
    elif isinstance(val, str):
        try: return float(val)
        except: return np.nan
    return np.nan

def calibrate_power_method(odds_series):
    """Remove bookmaker overround. Find k: sum(p^k)=1, then p_cal = p^k."""
    p = 1.0 / odds_series.replace(0, np.nan).dropna()
    if len(p) < 2: return p
    try:
        k = brentq(lambda k: np.sum(p**k)-1, 0.01, 20)
        return p**k
    except: return p / p.sum()

DRIVER_MAP = {
    'Max Verstappen':'VER','Liam Lawson':'LAW','Lando Norris':'NOR','Oscar Piastri':'PIA',
    'Charles Leclerc':'LEC','Lewis Hamilton':'HAM','George Russell':'RUS',
    'Andrea Kimi Antonelli':'ANT','Kimi Antonelli':'ANT','Fernando Alonso':'ALO',
    'Lance Stroll':'STR','Pierre Gasly':'GAS','Jack Doohan':'DOO',
    'Carlos Sainz Jr.':'SAI','Carlos Sainz':'SAI','Alexander Albon':'ALB',
    'Yuki Tsunoda':'TSU','Isack Hadjar':'HAD','Nico Hülkenberg':'HUL',
    'Nico Hulkenberg':'HUL','Gabriel Bortoleto':'BOR','Esteban Ocon':'OCO',
    'Oliver Bearman':'BEA','Franco Colapinto':'COL',
}
RACE_MAP = {
    'Avustralya GP':'Australian Grand Prix','Çin GP':'Chinese Grand Prix',
    'Japonya GP':'Japanese Grand Prix','Bahreyn GP':'Bahrain Grand Prix',
    'Suudi Arabistan GP':'Saudi Arabian Grand Prix','Miami GP':'Miami Grand Prix',
    'Emilia Romagna GP':'Emilia Romagna Grand Prix','Monako GP':'Monaco Grand Prix',
    'İspanya GP':'Spanish Grand Prix','Kanada GP':'Canadian Grand Prix',
    'Avusturya GP':'Austrian Grand Prix','Britanya GP':'British Grand Prix',
    'Belçika GP':'Belgian Grand Prix','Macaristan GP':'Hungarian Grand Prix',
    'Hollanda GP':'Dutch Grand Prix','İtalya GP':'Italian Grand Prix',
    'Azerbaycan GP':'Azerbaijan Grand Prix','Singapur GP':'Singapore Grand Prix',
    'ABD GP (Austin)':'United States Grand Prix','Meksika GP':'Mexican Grand Prix',
    'São Paulo GP':'São Paulo Grand Prix','Las Vegas GP':'Las Vegas Grand Prix',
    'Katar GP':'Qatar Grand Prix','Abu Dabi GP':'Abu Dhabi Grand Prix',
}
FEATS = ['Grid_Position', 'Quali_Time_Delta', 'Top_Speed_ST']
REAL_BASELINE = 3 / 20
print('Setup complete.')

---

## 2. Data Collection & Loading

Two independent data sources were collected and merged on `(race, driver_code)`:

| Source | Content | Collection Method |
|---|---|---|
| `FastF1` Python library | Grid position, qualifying time delta, speed trap | API pull, 2022–2024 & 2025 |
| Official F1 betting articles | Race win odds, podium odds | Web scraping + manual verification |

The merged dataset covers **2022, 2023 and 2024** seasons and forms the basis for training and evaluation.

In [ ]:
df = pd.read_csv('F1_Master_Merged_Data.csv')
print(f'Shape: {df.shape[0]} rows × {df.shape[1]} columns')
print(f'Seasons: {sorted(df["year"].unique())}')
print(f'Races:   {df["race"].nunique()} unique Grand Prix')
print(f'Drivers: {df["driver_code"].nunique()} unique driver codes')
print('\nColumn descriptions:')
descriptions = {
    'year':'Season','race':'Grand Prix name','driver_code':'FIA 3-letter code',
    'race_win_odds':'Decimal odds — race win','race_win_prob':'Implied prob — race win',
    'podium_odds':'Decimal odds — top-3 finish','podium_prob':'Implied prob — top-3 finish',
    'Grid_Position':'Starting grid position (1=pole)','Quali_Time_Delta':'Gap to pole (seconds)',
    'Top_Speed_ST':'Speed trap top speed in qualifying (km/h)',
}
for col,desc in descriptions.items():
    print(f'  {col:<20} {desc}')
df.head()

---

## 3. Descriptive Statistics & Data Quality

Before any analysis, the data's structure, distributions, and quality issues are examined.

In [ ]:
print('=== Descriptive Statistics ===')
print(df[FEATS + ['race_win_prob','podium_prob']].describe().round(3).to_string())
print('\n=== Missing Values ===')
missing = df.isnull().sum()
print(missing[missing > 0].to_string())

In [ ]:
# Missing value heatmap
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Left: missing counts
missing_pct = df.isnull().mean() * 100
missing_pct[missing_pct > 0].sort_values().plot(
    kind='barh', ax=axes[0], color='#e63946', edgecolor='white')
axes[0].set_title('Missing Values by Column (%)', fontsize=11)
axes[0].set_xlabel('Missing (%)')

# Right: heatmap of missingness by year
miss_by_year = df.groupby('year')[FEATS+['podium_prob','podium_odds']].apply(
    lambda g: g.isnull().mean()*100).T
sns.heatmap(miss_by_year, annot=True, fmt='.1f', cmap='Reds',
            linewidths=0.5, ax=axes[1], cbar_kws={'label':'Missing %'})
axes[1].set_title('Missing % by Feature and Season', fontsize=11)

plt.suptitle('Data Quality: Missing Values', fontsize=13, y=1.02)
plt.tight_layout(); plt.show()

In [ ]:
# Feature distributions
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
colors = ['#457b9d','#e63946','#2a9d8f']
for ax, feat, color in zip(axes, FEATS, colors):
    data = df[feat].dropna()
    ax.hist(data, bins=30, color=color, edgecolor='white', alpha=0.85)
    ax.axvline(data.mean(), color='black', linestyle='--', linewidth=1.5, label=f'Mean: {data.mean():.1f}')
    ax.axvline(data.median(), color='red', linestyle=':', linewidth=1.5, label=f'Median: {data.median():.1f}')
    ax.set_title(feat, fontsize=11)
    ax.set_xlabel(feat); ax.set_ylabel('Count')
    ax.legend(fontsize=8)
plt.suptitle('Distribution of Key Telemetry Features (2022-2024)', fontsize=13, y=1.02)
plt.tight_layout(); plt.show()
print('Observation: All three features are right-skewed — this guides the selection of appropriate statistical tests.')

---

## 4. Data Cleaning

Three quality issues were identified and corrected before analysis:

| Issue | Rows | Fix |
|---|---|---|
| Negative `Quali_Time_Delta` (sign flip during calculation) | 28 | `abs()` |
| `Top_Speed_ST` < 260 km/h at 2024 São Paulo (unit error: mph vs km/h) | 13 | Set to `NaN` |
| 2025 odds stored as x1000 (Turkish locale: dot = thousands separator) | 32 | Divide by 1000 |

In [ ]:
# Fix 1: negative time deltas
neg_before = (df['Quali_Time_Delta'] < 0).sum()
df['Quali_Time_Delta'] = df['Quali_Time_Delta'].abs()
print(f'Fixed {neg_before} negative Quali_Time_Delta values')

# Fix 2: São Paulo 2024 speed outliers
sp24_mask = (df['race']=='São Paulo Grand Prix')&(df['year']==2024)&(df['Top_Speed_ST']<260)
print(f'Fixed {sp24_mask.sum()} implausible Top_Speed_ST values at 2024 São Paulo GP')
df.loc[sp24_mask, 'Top_Speed_ST'] = np.nan

# Verify
print(f'Negative deltas remaining: {(df["Quali_Time_Delta"]<0).sum()}')
print(f'Sub-260 km/h speeds remaining: {(df["Top_Speed_ST"]<260).sum()} (2022-2023 Monaco, etc. expected)')

### 4.1 Podium Label

The label `podium = 1` is assigned to drivers who started in **Grid Position ≤ 3** AND had market-implied podium probability **> 35%**.

> **Important note on baselines:** Each race has exactly 3 podium positions for 20 drivers. The true random baseline for any podium prediction is **3/20 = 15%**, NOT the label rate (~10%). The label undercounts real podiums because it misses drivers who started P4+ and finished top-3. This limitation is acknowledged in Section 10.

In [ ]:
df['podium'] = ((df['Grid_Position']<=3) & (df['podium_prob'].fillna(0.5)>0.35)).astype(int)
total_races = df.groupby(['year','race']).ngroups
print(f'Label = 1: {df["podium"].sum()} ({df["podium"].mean():.2%})')
print(f'Label = 0: {(df["podium"]==0).sum()} ({1-df["podium"].mean():.2%})')
print(f'True random baseline (3/20): {REAL_BASELINE:.2%}')
print(f'Label misses ~{total_races*3 - df["podium"].sum()} real podiums (P4-20 finishers who reached top-3)')

---

## 5. Probability Calibration — Power Method

Bookmaker decimal odds contain a built-in profit margin (overround). Raw implied probabilities sum to more than 1 per race, which means using them directly would bias any EV calculation. The **Power Method** is applied to remove this margin:

$$P_{raw,i} = \\frac{1}{\\text{Odds}_i} \\qquad \\text{Find } k : \\sum_i (P_{raw,i})^k = 1 \\qquad P_{calibrated,i} = (P_{raw,i})^k$$

After calibration, probabilities sum to exactly 1 per race and represent fair-value market estimates.

In [ ]:
for col_odds, col_cal in [('race_win_odds','win_prob_cal'),('podium_odds','pod_prob_cal')]:
    df[col_cal] = np.nan
    for (yr,race),g in df.groupby(['year','race']):
        s = g[col_odds].dropna()
        if len(s)>=2:
            c=calibrate_power_method(s); df.loc[c.index,col_cal]=c.values

overround = (
    df.dropna(subset=['race_win_odds'])
    .groupby(['year','race'])
    .apply(lambda g: (1/g['race_win_odds']).sum())
    .reset_index(name='overround')
)
print('Overround statistics by season:')
print(overround.groupby('year')['overround'].agg(['mean','min','max']).round(3).to_string())

# Visualise
fig, axes = plt.subplots(1,2,figsize=(13,4))
sns.boxplot(data=overround,x='year',y='overround',palette='muted',ax=axes[0])
axes[0].axhline(1.0,color='red',linestyle='--',linewidth=1.5,label='No margin')
axes[0].set_title('Overround by Season — Race Win Market',fontsize=11); axes[0].legend()

# Before vs after calibration
sample = df[(df['race']=='Bahrain Grand Prix')&(df['year']==2023)].dropna(subset=['race_win_odds','win_prob_cal'])
x = np.arange(len(sample))
axes[1].bar(x-0.2, sample['race_win_prob'],   0.4, label='Raw implied', color='#e63946', alpha=0.7)
axes[1].bar(x+0.2, sample['win_prob_cal'], 0.4, label='Calibrated',  color='#457b9d', alpha=0.7)
axes[1].set_xticks(x); axes[1].set_xticklabels(sample['driver_code'],rotation=45,fontsize=8)
axes[1].set_title('2023 Bahrain: Raw vs Calibrated Win Prob',fontsize=11)
axes[1].set_ylabel('Probability'); axes[1].legend()

plt.tight_layout(); plt.show()
print('The bookmaker margin averages 8-10% — calibration is essential before any EV comparison.')

---

## 6. Exploratory Data Analysis (EDA)

With clean data and calibrated probabilities, relationships between telemetry features and podium outcomes are explored. The goal is to identify patterns that motivate the formal hypothesis tests.

In [ ]:
# Correlation heatmap
feats_corr = ['Grid_Position','Quali_Time_Delta','Top_Speed_ST','podium','pod_prob_cal']
corr = df[feats_corr].corr()
plt.figure(figsize=(8,6))
sns.heatmap(corr,annot=True,cmap='coolwarm',vmin=-1,vmax=1,fmt='.2f',linewidths=0.5)
plt.title('Correlation Matrix: Telemetry Features vs Podium',fontsize=13)
plt.tight_layout(); plt.show()
print('Key patterns:')
print(f'  Grid_Position ↔ podium:      {corr.loc["Grid_Position","podium"]:+.3f}  (strong negative — front = better)')
print(f'  Quali_Time_Delta ↔ podium:   {corr.loc["Quali_Time_Delta","podium"]:+.3f}  (negative — smaller gap = better)')
print(f'  Top_Speed_ST ↔ podium:       {corr.loc["Top_Speed_ST","podium"]:+.3f}  (weak positive)')

In [ ]:
fig, axes = plt.subplots(1,3,figsize=(16,5))

# Grid band podium rate
df_g = df.dropna(subset=['Grid_Position']).copy()
df_g['grid_bin'] = pd.cut(df_g['Grid_Position'],bins=[0,3,6,10,15,20],
                           labels=['P1-3','P4-6','P7-10','P11-15','P16-20'])
rate = df_g.groupby('grid_bin',observed=True)['podium'].mean()*100
bars = axes[0].bar(rate.index,rate.values,color='#e63946',edgecolor='white',width=0.6)
for b in bars:
    axes[0].annotate(f'{b.get_height():.1f}%',(b.get_x()+b.get_width()/2,b.get_height()+0.5),ha='center',fontsize=10)
axes[0].axhline(REAL_BASELINE*100,color='navy',linestyle='--',linewidth=1.5,label=f'True baseline ({REAL_BASELINE:.0%})')
axes[0].set_title('Podium Rate by Grid Band',fontsize=11); axes[0].set_ylabel('Rate (%)'); axes[0].legend(fontsize=8)

# Quali_Time_Delta distribution
df_d = df.dropna(subset=['Quali_Time_Delta']).copy()
df_d['Outcome'] = df_d['podium'].map({1:'Podium',0:'No Podium'})
sns.kdeplot(data=df_d,x='Quali_Time_Delta',hue='Outcome',
            palette={'Podium':'#e63946','No Podium':'#457b9d'},
            fill=True,alpha=0.35,linewidth=2,ax=axes[1])
axes[1].set_xlim(-0.5,10); axes[1].set_title('Quali Time Delta Distribution',fontsize=11)
axes[1].set_xlabel('Time Delta to Pole (seconds)')
for label,color in [('Podium','#e63946'),('No Podium','#457b9d')]:
    m=df_d[df_d['Outcome']==label]['Quali_Time_Delta'].mean()
    axes[1].axvline(m,color=color,linestyle='--',linewidth=1.5,label=f'{label} mean: {m:.2f}s')
axes[1].legend(fontsize=8)

# Top speed
df_s = df.dropna(subset=['Top_Speed_ST']).copy()
df_s['Outcome'] = df_s['podium'].map({1:'Podium',0:'No Podium'})
sns.kdeplot(data=df_s,x='Top_Speed_ST',hue='Outcome',
            palette={'Podium':'#e63946','No Podium':'#457b9d'},
            fill=True,alpha=0.35,linewidth=2,ax=axes[2])
axes[2].set_title('Top Speed Distribution',fontsize=11); axes[2].set_xlabel('Speed Trap (km/h)')
for label,color in [('Podium','#e63946'),('No Podium','#457b9d')]:
    m=df_s[df_s['Outcome']==label]['Top_Speed_ST'].mean()
    axes[2].axvline(m,color=color,linestyle='--',linewidth=1.5,label=f'{label}: {m:.1f}')
axes[2].legend(fontsize=8)

plt.suptitle('EDA: Three Telemetry Features vs Podium Outcome',fontsize=13,y=1.02)
plt.tight_layout(); plt.show()
print('Observations:')
print('  1. P1-3 starters finish on podium dramatically more often → motivates H1')
print('  2. Podium drivers are clustered near 0.0s delta; non-podium spread widely → motivates H3')
print('  3. Top speed distributions overlap heavily → suggests H2 may not reject')

---

## 7. Hypothesis Testing

Based on the EDA patterns, three hypotheses are formally tested. Before selecting a test, the normality assumption is verified.

### 7.1 Normality Check — Shapiro-Wilk Test

Many parametric tests (e.g. t-test) assume normally distributed data. This assumption is verified before proceeding.

In [ ]:
print('Shapiro-Wilk normality test (H₀: data is normally distributed):')
print(f'{"Feature":<28} {"W":>8} {"p-value":>12} {"Normal?":>10}')
print('-'*60)
for feat in FEATS:
    sample = df[feat].dropna().sample(min(500,len(df[feat].dropna())),random_state=42)
    w,p = stats.shapiro(sample)
    normal = 'Yes' if p>=0.05 else 'No — non-normal'
    print(f'{feat:<28} {w:>8.4f} {p:>12.4e} {normal:>16}')
print()
print('Conclusion: All three features are non-normally distributed (p << 0.05).')
print('→ The Mann-Whitney U test is used for all hypotheses.')
print('  (Mann-Whitney is a non-parametric test that makes no normality assumption.)')

### 7.2 Hypothesis 1 — Does Grid Position Predict Win Probability?

The EDA showed P1-3 starters have dramatically higher podium rates. This is formally tested below.

- **H₀:** There is no difference in calibrated win probability between Top-3 qualifiers and the rest  
- **H₁:** Top-3 qualifiers have significantly higher calibrated win probability  
- **Test:** Mann-Whitney U (one-sided, alternative = 'greater')

In [ ]:
h1_df = df.dropna(subset=['Grid_Position','win_prob_cal'])
top3  = h1_df[h1_df['Grid_Position']<=3]['win_prob_cal']
rest  = h1_df[h1_df['Grid_Position']>3]['win_prob_cal']
u_h1, p_h1 = stats.mannwhitneyu(top3, rest, alternative='greater')

print('='*55)
print('H1 — Grid Position vs Win Probability (Mann-Whitney U)')
print('='*55)
print(f'  Top-3  — N={len(top3)},  Median: {top3.median():.4f}, Mean: {top3.mean():.4f}')
print(f'  Rest   — N={len(rest)}, Median: {rest.median():.4f}, Mean: {rest.mean():.4f}')
print(f'  U = {u_h1:.0f} | p = {p_h1:.4e}')
print(f'  → ✅ REJECT H₀ — Top-3 qualifiers have {top3.mean()/rest.mean():.1f}× higher win probability')

### 7.3 Hypothesis 2 — Does Top Speed Predict Podium Outcome?

The EDA revealed heavy overlap in top speed distributions.

- **H₀:** There is no difference in top speed between podium and non-podium drivers  
- **H₁:** Podium drivers have significantly higher top speed  
- **Test:** Mann-Whitney U (one-sided, alternative = 'greater')

In [ ]:
h2_df     = df.dropna(subset=['Top_Speed_ST','podium'])
pod_spd   = h2_df[h2_df['podium']==1]['Top_Speed_ST']
nopod_spd = h2_df[h2_df['podium']==0]['Top_Speed_ST']
u_h2, p_h2 = stats.mannwhitneyu(pod_spd, nopod_spd, alternative='greater')

print('='*55)
print('H2 — Top Speed vs Podium Outcome (Mann-Whitney U)')
print('='*55)
print(f'  Podium    — N={len(pod_spd)}, Median: {pod_spd.median():.1f} km/h, Mean: {pod_spd.mean():.1f}')
print(f'  No Podium — N={len(nopod_spd)}, Median: {nopod_spd.median():.1f} km/h, Mean: {nopod_spd.mean():.1f}')
print(f'  Difference: {pod_spd.mean()-nopod_spd.mean():.1f} km/h')
print(f'  U = {u_h2:.0f} | p = {p_h2:.4f}')
if p_h2 >= 0.05:
    print(f'  → ❌ FAIL TO REJECT H₀ — {pod_spd.mean()-nopod_spd.mean():.1f} km/h gap is not significant.')
    print('  Top speed alone is not a reliable predictor of podium outcomes.')

### 7.4 Hypothesis 3 — Does Qualifying Time Delta Predict Podium Outcome?

The EDA revealed a striking difference: podium drivers cluster near 0.0s delta while non-podium drivers spread widely. This pattern warrants formal testing.

- **H₀:** There is no difference in Qualifying Time Delta between podium and non-podium drivers  
- **H₁:** Podium drivers have significantly smaller time delta to pole  
- **Test:** Mann-Whitney U (one-sided, alternative = 'less' — podium drivers have SMALLER delta)

In [ ]:
h3_df      = df.dropna(subset=['Quali_Time_Delta','podium'])
pod_delta  = h3_df[h3_df['podium']==1]['Quali_Time_Delta']
nopod_delta= h3_df[h3_df['podium']==0]['Quali_Time_Delta']
u_h3, p_h3 = stats.mannwhitneyu(pod_delta, nopod_delta, alternative='less')

print('='*55)
print('H3 — Quali Time Delta vs Podium Outcome (Mann-Whitney U)')
print('='*55)
print(f'  Podium    — N={len(pod_delta)}, Median: {pod_delta.median():.3f}s, Mean: {pod_delta.mean():.3f}s')
print(f'  No Podium — N={len(nopod_delta)}, Median: {nopod_delta.median():.3f}s, Mean: {nopod_delta.mean():.3f}s')
print(f'  U = {u_h3:.0f} | p = {p_h3:.4e}')
print(f'  → ✅ REJECT H₀ — Podium drivers are {nopod_delta.mean()/pod_delta.mean():.0f}× closer to pole than non-podium drivers.')

In [ ]:
# Summary visualisation
fig, axes = plt.subplots(1,3,figsize=(15,5))

# H1
plot_h1 = pd.DataFrame({'Win Prob':pd.concat([top3,rest]),'Group':['Top-3']*len(top3)+['Rest']*len(rest)})
sns.boxplot(data=plot_h1,x='Group',y='Win Prob',palette={'Top-3':'#e63946','Rest':'#457b9d'},ax=axes[0])
axes[0].set_title(f'H1: Win Prob by Grid Band\nU={u_h1:.0f}, p={p_h1:.2e} ✅ Reject',fontsize=10)

# H2
plot_h2 = pd.DataFrame({'Top Speed':pd.concat([pod_spd,nopod_spd]),'Outcome':['Podium']*len(pod_spd)+['No Podium']*len(nopod_spd)})
sns.boxplot(data=plot_h2,x='Outcome',y='Top Speed',palette={'Podium':'#e63946','No Podium':'#457b9d'},ax=axes[1])
axes[1].set_title(f'H2: Top Speed by Outcome\nU={u_h2:.0f}, p={p_h2:.3f} ❌ Not Significant',fontsize=10)
axes[1].set_ylabel('Speed Trap (km/h)')

# H3 — log scale for visibility
plot_h3 = pd.DataFrame({'Quali Delta':pd.concat([pod_delta,nopod_delta]),'Outcome':['Podium']*len(pod_delta)+['No Podium']*len(nopod_delta)})
sns.boxplot(data=plot_h3,x='Outcome',y='Quali Delta',palette={'Podium':'#e63946','No Podium':'#457b9d'},ax=axes[2])
axes[2].set_title(f'H3: Time Delta by Outcome\nU={u_h3:.0f}, p={p_h3:.2e} ✅ Reject',fontsize=10)
axes[2].set_ylabel('Delta to Pole (seconds)')

plt.suptitle('Hypothesis Test Results',fontsize=13,y=1.02)
plt.tight_layout(); plt.show()

print('\nHypothesis Summary:')
print(f'  H1 (Grid ≤3 → win prob):       U={u_h1:.0f}, p={p_h1:.2e}  ✅ Reject H₀')
print(f'  H2 (Top speed → podium):        U={u_h2:.0f}, p={p_h2:.4f}   ❌ Fail to Reject H₀')
print(f'  H3 (Time delta → podium):       U={u_h3:.0f}, p={p_h3:.2e}  ✅ Reject H₀')

**Key finding from hypothesis testing:**
- **Grid Position and Qualifying Time Delta** are both statistically significant predictors of podium outcomes (p < 0.001).
- **Top Speed alone is NOT significant** — the ~1 km/h difference is too small to be meaningful.
- These findings directly inform feature selection for the ML models.

---

## 8. Machine Learning Models

Two classification models are trained to quantify the predictive power of telemetry features. Based on the hypothesis tests, Grid Position (H1) and Time Delta (H3) are expected to dominate feature importance, while Top Speed (H2) is expected to contribute minimally.

**Setup:**
- Train set: 2022–2023 seasons
- Test set: 2024 season (true temporal out-of-sample — no future data seen during training)
- StandardScaler fit on train only, applied to test
- `class_weight='balanced'` used in all models to handle the ~10:90 class imbalance

**Models compared:**
- **Dummy Classifier** — random baseline (no predictive power)
- **Logistic Regression** — linear, interpretable
- **Random Forest** — non-linear, feature interactions
- **Ensemble** — average of LR and RF probabilities

In [ ]:
df_ml  = df.dropna(subset=FEATS+['podium','podium_odds'])
train  = df_ml[df_ml['year']<=2023]
test   = df_ml[df_ml['year']==2024]
X_tr, y_tr = train[FEATS], train['podium']
X_te, y_te = test[FEATS],  test['podium']

scaler   = StandardScaler()
X_tr_sc  = scaler.fit_transform(X_tr)
X_te_sc  = scaler.transform(X_te)

print(f'Train: {len(X_tr)} rows | Podium rate: {y_tr.mean():.2%}')
print(f'Test:  {len(X_te)} rows | Podium rate: {y_te.mean():.2%}')
print(f'True random baseline: {REAL_BASELINE:.2%}')

### 8.1 Cross-Validation

Before evaluating on the 2024 held-out set, **5-fold stratified cross-validation** is applied on the training set (2022-2023) to assess generalisation and detect potential overfitting.

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

lr_model = LogisticRegression(class_weight='balanced', max_iter=500, random_state=42)
rf_model = RandomForestClassifier(n_estimators=300, class_weight='balanced', max_depth=5, random_state=42)

lr_cv = cross_val_score(lr_model, X_tr_sc, y_tr, cv=cv, scoring='roc_auc')
rf_cv = cross_val_score(rf_model, X_tr,    y_tr, cv=cv, scoring='roc_auc')

print('5-Fold Cross-Validation AUC (on 2022-2023 train set):')
print(f'  Logistic Regression: {lr_cv.mean():.4f} ± {lr_cv.std():.4f}')
print(f'  Random Forest:       {rf_cv.mean():.4f} ± {rf_cv.std():.4f}')
print()
print('Low standard deviation confirms models are stable and not overfitting to any single fold.')

In [ ]:
# Train final models on full train set
dummy_model = DummyClassifier(strategy='stratified', random_state=42)
dummy_model.fit(X_tr, y_tr)
lr_model.fit(X_tr_sc, y_tr)
rf_model.fit(X_tr, y_tr)

dummy_p = dummy_model.predict_proba(X_te)[:,1]
lr_p    = lr_model.predict_proba(X_te_sc)[:,1]
rf_p    = rf_model.predict_proba(X_te)[:,1]
ens_p   = (lr_p + rf_p) / 2

print('Test Set Performance (2024 out-of-sample):')
print(f'  {"Model":<22} {"ROC-AUC":>8}  {"PR-AUC":>8}')
print(f'  {"-"*42}')
print(f'  {"Dummy (baseline)":<22} {roc_auc_score(y_te,dummy_p):>8.4f}  {average_precision_score(y_te,dummy_p):>8.4f}')
print(f'  {"Logistic Regression":<22} {roc_auc_score(y_te,lr_p):>8.4f}  {average_precision_score(y_te,lr_p):>8.4f}')
print(f'  {"Random Forest":<22} {roc_auc_score(y_te,rf_p):>8.4f}  {average_precision_score(y_te,rf_p):>8.4f}')
print(f'  {"Ensemble":<22} {roc_auc_score(y_te,ens_p):>8.4f}  {average_precision_score(y_te,ens_p):>8.4f}')
print()
print('PR-AUC is particularly informative for imbalanced classes (label rate ~10%).')
print('The models score ~0.82 PR-AUC vs 0.10 for random — a strong result.')

In [ ]:
# Feature importance & coefficients
print('Random Forest Feature Importances (how much each feature contributes):')
for f,i in sorted(zip(FEATS,rf_model.feature_importances_),key=lambda x:-x[1]):
    print(f'  {f:<28} {i:.4f}  {chr(9608)*int(i*45)}')
print()
print('Logistic Regression Coefficients (direction of effect):')
for f,c in zip(FEATS,lr_model.coef_[0]):
    direction = 'higher = less likely podium' if c<0 else 'higher = more likely podium'
    print(f'  {f:<28} {c:+.4f}  ({direction})')
print()
print('Consistent with hypotheses: Grid_Position and Quali_Time_Delta dominate.')
print('Negative coefficients confirm: lower grid position and smaller time delta = higher podium chance.')

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14,11))

# Feature importance
fi = pd.Series(rf_model.feature_importances_,index=FEATS).sort_values()
fi.plot(kind='barh',ax=axes[0,0],color='#457b9d',edgecolor='white')
axes[0,0].set_title('RF Feature Importance',fontsize=11); axes[0,0].set_xlabel('Importance')

# ROC curves
for prob,label,color in [
    (dummy_p,f'Dummy (AUC={roc_auc_score(y_te,dummy_p):.3f})','#adb5bd'),
    (lr_p,   f'LR    (AUC={roc_auc_score(y_te,lr_p):.3f})',  '#457b9d'),
    (rf_p,   f'RF    (AUC={roc_auc_score(y_te,rf_p):.3f})',  '#e63946'),
    (ens_p,  f'Ens   (AUC={roc_auc_score(y_te,ens_p):.3f})', '#2a9d8f'),
]:
    fpr,tpr,_ = roc_curve(y_te,prob)
    axes[0,1].plot(fpr,tpr,label=label,linewidth=2)
axes[0,1].plot([0,1],[0,1],'k--',linewidth=1)
axes[0,1].set_title('ROC Curves — 2024 Test Set',fontsize=11)
axes[0,1].set_xlabel('FPR'); axes[0,1].set_ylabel('TPR'); axes[0,1].legend(fontsize=8)

# Precision-Recall curves (better for imbalanced data)
for prob,label,color in [
    (dummy_p,f'Dummy (PR-AUC={average_precision_score(y_te,dummy_p):.3f})','#adb5bd'),
    (lr_p,   f'LR    (PR-AUC={average_precision_score(y_te,lr_p):.3f})',  '#457b9d'),
    (rf_p,   f'RF    (PR-AUC={average_precision_score(y_te,rf_p):.3f})',  '#e63946'),
    (ens_p,  f'Ens   (PR-AUC={average_precision_score(y_te,ens_p):.3f})', '#2a9d8f'),
]:
    prec_c,rec_c,_ = precision_recall_curve(y_te,prob)
    axes[1,0].plot(rec_c,prec_c,label=label,linewidth=2,color=color)
axes[1,0].axhline(y_te.mean(),color='gray',linestyle='--',linewidth=1,label=f'Baseline ({y_te.mean():.2%})')
axes[1,0].set_title('Precision-Recall Curves — 2024 Test',fontsize=11)
axes[1,0].set_xlabel('Recall'); axes[1,0].set_ylabel('Precision'); axes[1,0].legend(fontsize=8)

# Confusion matrix
ConfusionMatrixDisplay.from_predictions(y_te,(ens_p>=0.30).astype(int),
    display_labels=['No Podium','Podium'],cmap='Blues',ax=axes[1,1])
axes[1,1].set_title('Ensemble Confusion Matrix (threshold=0.30)',fontsize=11)

plt.suptitle('Model Evaluation — 2024 Out-of-Sample Test',fontsize=13,y=1.01)
plt.tight_layout(); plt.show()

---

## 9. Expected Value (EV) Simulation

With model probabilities computed, **value bets** are identified — situations where the model estimates a higher podium probability than the market implies.

$$\\text{EV} = (P_{\\text{model}} \\times \\text{Decimal Odds}) - 1$$

| EV | Meaning |
|---|---|
| EV > 0 | Model sees higher probability than market → potential value |
| EV < 0 | Market overestimates driver → no value |

Bets are flagged when **EV > 0.05** (design choice: model probability exceeds market implied probability by at least 5 percentage points). A bet with EV = 0.20 implies an expected return of +0.20 units per 1 unit staked over the long run.

> **On precision:** The correct baseline is **15% (3/20)** — three podium places per 20-driver field. Achieving 37% precision means the model's selections are **2.5× more accurate than randomly selecting a driver**.

In [ ]:
te_ev = test.copy().reset_index(drop=True)
te_ev['prob_ens'] = ens_p
te_ev['ev']       = te_ev['prob_ens'] * te_ev['podium_odds'] - 1
te_ev['value_bet'] = te_ev['ev'] > 0.05

n_vb  = te_ev['value_bet'].sum()
prec  = te_ev[te_ev['value_bet']]['podium'].mean()

print(f'Test rows:                    {len(te_ev)}')
print(f'Value bets flagged (EV>0.05): {n_vb}')
print(f'VB precision (on label=1):    {prec:.2%}')
print(f'True random baseline (3/20):  {REAL_BASELINE:.2%}')
print(f'Model vs baseline:            {prec/REAL_BASELINE:.1f}× better than random')
print()
cols = ['race','driver_code','Grid_Position','podium_odds','prob_ens','ev','podium']
print('Top 15 Value Bets — 2024 Backtest:')
print(te_ev[te_ev['value_bet']][cols].sort_values('ev',ascending=False).head(15).to_string(index=False))

In [ ]:
fig, axes = plt.subplots(1,2,figsize=(13,5))

te_ev['ev'].clip(-1.5,5).hist(bins=40,ax=axes[0],color='#457b9d',edgecolor='white')
axes[0].axvline(0.05,color='red',linestyle='--',linewidth=2,label='Value bet threshold')
axes[0].axvline(0,color='orange',linestyle=':',linewidth=1.5,label='EV=0')
axes[0].set_title('EV Distribution — 2024',fontsize=11); axes[0].set_xlabel('EV'); axes[0].legend()

te_ev['market_implied'] = 1/te_ev['podium_odds']
te_ev['Outcome'] = te_ev['podium'].map({1:'Podium',0:'No Podium'})
for outcome,color,marker in [('Podium','#e63946','o'),('No Podium','#457b9d','x')]:
    s=te_ev[te_ev['Outcome']==outcome]
    axes[1].scatter(s['market_implied'],s['prob_ens'],c=color,marker=marker,alpha=0.5,s=25,label=outcome)
axes[1].plot([0,1],[0,1],'k--',linewidth=1.5,label='Model = Market')
axes[1].fill_between([0,1],[0.05,1.05],[0,1],alpha=0.08,color='green',label='Value zone')
axes[1].set_xlim(0,0.8); axes[1].set_ylim(0,1)
axes[1].set_title('Model Prob vs Market Implied Prob',fontsize=11)
axes[1].set_xlabel('Market Implied (1/odds)'); axes[1].legend(fontsize=9)

plt.suptitle('EV Simulation — 2024 Backtest Results',fontsize=13,y=1.02)
plt.tight_layout(); plt.show()

---

## 10. 2025 Season — Live Predictions

The trained model is applied directly to the 2025 season qualifying data collected from FastF1, with no retraining.

In [ ]:
# Load 2025 odds (with locale fix)
doc = Document('F1_2025_Odds.numbers')
raw = [[cell.value for cell in row] for row in doc.sheets[0].tables[0].iter_rows()]
odds_raw = pd.DataFrame(raw[1:],columns=raw[0])
odds_raw['race']          = odds_raw['Yarış'].map(RACE_MAP)
odds_raw['driver_code']   = odds_raw['Pilot'].map(DRIVER_MAP)
odds_raw['race_win_odds'] = odds_raw['Bahis Kazanma Oranı'].apply(fix_odds)
odds_raw['podium_odds']   = odds_raw['Bahis Podyum Oranı'].apply(fix_odds)

# Load 2025 telemetry
tel = pd.read_csv('F1_2025_Grid_Quali_Speed_Data.csv')
tel = tel.rename(columns={'Yaris':'race','Pilot':'full_name'})
tel['race'] = tel['race'].replace({'Mexico City Grand Prix':'Mexican Grand Prix'})
tel['driver_code'] = tel['full_name'].map(DRIVER_MAP)
tel['Quali_Time_Delta'] = tel['Quali_Time_Delta'].abs()

# Merge
df25 = pd.merge(tel[['race','driver_code','Grid_Position','Quali_Time_Delta','Top_Speed_ST']],
                odds_raw[['race','driver_code','race_win_odds','podium_odds']],
                on=['race','driver_code'],how='left')

# Calibrate odds
df25['win_prob_cal']=np.nan; df25['pod_prob_cal']=np.nan
for race,g in df25.groupby('race'):
    wo=g['race_win_odds'].dropna()
    if len(wo)>=2: c=calibrate_power_method(wo); df25.loc[c.index,'win_prob_cal']=c.values
    po=g['podium_odds'].dropna()
    if len(po)>=2: c=calibrate_power_method(po); df25.loc[c.index,'pod_prob_cal']=c.values

# Predict
df25_ml = df25.dropna(subset=FEATS).copy()
X25 = df25_ml[FEATS]; X25_sc = scaler.transform(X25)
df25_ml['prob_lr']  = lr_model.predict_proba(X25_sc)[:,1]
df25_ml['prob_rf']  = rf_model.predict_proba(X25)[:,1]
df25_ml['prob_ens'] = (df25_ml['prob_lr']+df25_ml['prob_rf'])/2
df25_ml['ev_podium']= df25_ml['prob_ens']*df25_ml['podium_odds']-1
df25_ml['value_bet']= df25_ml['ev_podium']>0.05

print(f'2025: {len(df25_ml)} rows | {df25_ml["race"].nunique()} races | {df25_ml["value_bet"].sum()} value bets')
print('\nTop 20 Value Bets — 2025 Season:')
vb_cols=['race','driver_code','Grid_Position','Quali_Time_Delta','podium_odds','prob_ens','ev_podium']
print(df25_ml[df25_ml['value_bet']][vb_cols].sort_values('ev_podium',ascending=False).head(20).to_string(index=False))

In [ ]:
# Best EV per race summary
best = (df25_ml.dropna(subset=['ev_podium']).sort_values('ev_podium',ascending=False)
        .groupby('race',sort=False).first().reset_index()
        [['race','driver_code','Grid_Position','podium_odds','prob_ens','ev_podium','value_bet']]
        .sort_values('race'))
print('Best EV opportunity per race — 2025 full season:')
print(best.to_string(index=False))
df25_ml.to_csv('F1_2025_Full_Predictions.csv',index=False)
print('\nExported: F1_2025_Full_Predictions.csv')

---

## 11. Limitations & Future Work

### Known Limitations

| # | Issue | Impact | Status |
|---|---|---|---|
| 1 | **Circular label**: `Grid_Position` in both label definition and model features | AUC partially inflated (~0.98 vs ~0.87 with market-only label) | Acknowledged — actual race results unavailable via API |
| 2 | **Label undercount**: misses P4-20 drivers who finish top-3 | Label rate 10% vs real baseline 15% | Use 15% as correct baseline throughout |
| 3 | **EV threshold**: 0.05 is a design choice | Different thresholds → different trade-offs | Should be optimised on validation set |
| 4 | **Small sample**: 67 races | Betting edge estimates need larger n | Add seasons as they complete |
| 5 | **2025 odds locale fix**: Turkish decimal separator | Corrected automatically by ÷1000 | Manually validated |
| 6 | **No hyperparameter tuning**: default max_depth=5 | May not be optimal | Add GridSearchCV |

### Next Steps

- Obtain actual race results and retrain with true podium labels
- Add sector times and tyre compound as additional features
- Optimise EV threshold using calibration curves
- Implement Kelly criterion staking for proper bankroll management
- Apply SHAP values for per-prediction interpretability

---

## 12. Conclusions

### Results Summary

| | Result |
|---|---|
| **H1** Grid Position → win probability | U-test, p=7.96e-23 **✅ Reject H₀** — Top-3 qualifiers have 11× higher win probability |
| **H2** Top Speed → podium | U-test, p≈0.47 **❌ Fail to Reject H₀** — 1 km/h gap not significant |
| **H3** Quali Time Delta → podium | U-test, p=1.76e-66 **✅ Reject H₀** — Podium drivers 14× closer to pole |
| **5-fold CV AUC** (train, 2022-2023) | LR: 0.979 ± 0.011 · RF: 0.979 ± 0.010 — stable, not overfitting |
| **Test AUC** (2024 out-of-sample) | Ensemble: **0.9809** vs Dummy: 0.52 |
| **PR-AUC** (imbalanced metric) | Ensemble: **0.826** vs Dummy: 0.106 |
| **EV backtest precision** (2024) | **37%** vs **15% true baseline** = **2.5× better than random** |
| **2025 value bets** | **116** flagged across all 24 races |

### Narrative Summary

The analysis began with the collection of two independent data sources — qualifying telemetry from FastF1 and bookmaker odds from official F1 betting articles. After cleaning and merging, normality checks on all features showed non-normal distributions, leading to the adoption of Mann-Whitney U tests throughout.

**Hypothesis testing** confirmed that Grid Position (H1) and Qualifying Time Delta (H3) are strongly significant predictors of podium outcomes, while Top Speed (H2) is not significant on its own. This aligned with the EDA patterns and directly informed feature selection.

**Machine learning models** (Logistic Regression and Random Forest) trained on 2022-2023 data achieved **AUC 0.98** and **PR-AUC 0.83** on the held-out 2024 season. Cross-validation confirmed stability with ± 0.01 standard deviation across folds. Feature importances confirmed the hypothesis findings: Grid Position (55%) and Time Delta (41%) dominate.

**The EV simulation** demonstrates that when the model identifies a value bet, it achieves **37% precision — 2.5× the true baseline of 15%**. Applied to the full 2025 season, 116 value betting opportunities were flagged across 24 races.